# 02 - SPLADE: Sparse Retrieval

This notebook walks you through:

1. Installing the extra dependencies needed for SPLADE
2. Indexing all passages (encoding + storing sparse vectors in PostgreSQL)
3. Running search queries with SPLADE
4. Evaluating retrieval quality against the `qrels` ground truth

> **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
> `passages`, `queries`, and `qrels` tables are populated.

In [ ]:
import sys
from pathlib import Path

# Resolve project root
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## 1) Install extra dependencies

SPLADE requires `transformers` and `torch` on top of the base project requirements.

In [3]:
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(project_root / "requirements.txt")],
    check=True,
)

CompletedProcess(args=['C:\\Users\\Louis\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', '-r', 'C:\\Users\\Louis\\IR-M1\\Projet_Big_Data_IR-main\\requirements.txt'], returncode=0)

## 2) Index passages with SPLADE

This step encodes every passage in the `passages` table and stores the resulting sparse vector (`{token: weight}` JSON) in the `splade` table.

**Estimated time:** ~1-3 min per 10 000 passages on CPU, much faster on GPU.

You can re-run safely — already-indexed passages are skipped.

In [ ]:
from src.splade.indexer import index_passages

# Adjust batch sizes depending on your machine's RAM / VRAM
index_passages(batch_size=64, encoding_batch=32)

### Verify indexing

In [4]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM splade", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample SPLADE entry (first passage):")
    sample = pd.read_sql_query(
        "SELECT passage_id, term_weights FROM splade ORDER BY passage_id LIMIT 1",
        conn
    )
    display(sample)
finally:
    conn.close()

2026-03-07 13:37:49,855 - INFO - Database connection established successfully.


Indexed passages: 676193

Sample SPLADE entry (first passage):


,passage_id,term_weights
0,1,"{'%': 0.6214, 'br': 1.0901, 'hq': 0.2624, 'np'..."


## 3) Search with SPLADE

Two retrieval strategies:

- **GIN-based** (`search_gin`): uses the PostgreSQL GIN index to find candidate passages that share at least one term with the query, then scores them with a dot-product.
- **Brute-force** (`search_bruteforce`): scores every indexed passage. Slower but useful as a sanity check.

In [5]:
from src.splade.search import search_gin, search_bruteforce

query = "what is the speed of light"

print(f"Query: '{query}'\n")
results = search_gin(query, top_k=10)

for i, r in enumerate(results, 1):
    print(f"[{i}] Score={r['score']:.4f}  Passage#{r['passage_id']}")
    print(f"    {r['text'][:150]}...\n")

C:\Users\Louis\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-07 13:38:21,425 - INFO - Database connection established successfully.
2026-03-07 13:38:21,426 - INFO - Loading SPLADE model 'naver/splade-cocondenser-ensembledistil' on cpu ...


Query: 'what is the speed of light'



2026-03-07 13:38:21,830 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-07 13:38:21,834 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/config.json "HTTP/1.1 200 OK"
2026-03-07 13:38:21,942 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-07 13:38:21,948 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-07 13:38:22,052 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocondenser-ensembledistil/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Fo

[1] Score=22.6496  Passage#427302
    View full size image. The speed of light in a vacuum is 186,282 miles per second (299,792 kilometers per second), and in theory nothing can travel fas...

[2] Score=22.3885  Passage#461693
    View full size image. The speed of light in a vacuum is 186,282 miles per second (299,792 kilometers per second), and in theory nothing can travel fas...

[3] Score=21.5042  Passage#427307
    The speed of light in a medium is the speed of light in a vacuum divided by the index of refract … ion of the material. The index of refraction of gla...

[4] Score=21.4101  Passage#363195
    1 The speed of light, c is actually a constant and is roughly equal to 3.00x10 8 meters per second. 2  It is squared because of the basic properties o...

[5] Score=21.2464  Passage#614384
    Best Answer: Light is the physical speed limit of the Universe (as far as we know) (scientists take great pains not to declare anything conclusively b...

[6] Score=21.2437  Passage#427304
 

In [6]:
# Try your own queries
for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    results = search_gin(q, top_k=3)
    print(f"\nQuery: '{q}'")
    for i, r in enumerate(results, 1):
        print(f"  [{i}] {r['score']:.4f} — {r['text'][:120]}...")

2026-03-07 13:39:20,458 - INFO - Database connection established successfully.
2026-03-07 13:39:54,521 - INFO - SPLADE search: 475230 candidates, top-3 returned (encode=56ms, retrieval=31626ms, scoring=2380ms, total=34062ms)
2026-03-07 13:39:56,971 - INFO - Database connection established successfully.



Query: 'how old is the earth'
  [1] 20.0764 — 6 billion years ago. No, the Earth is only 4.6 billion years old. Amphibians appeared on Earth during the late Devonian ...
  [2] 19.7459 — Geologic and Biological Timeline of the Earth. Astronomical and geological evidence indicates that the Universe is appro...
  [3] 19.5356 — Scientists have found the oldest known rocks on Earth. They are 4.28 billion years old, making them 250 million years mo...


2026-03-07 13:40:15,715 - INFO - SPLADE search: 292884 candidates, top-3 returned (encode=56ms, retrieval=17178ms, scoring=1510ms, total=18744ms)
2026-03-07 13:40:17,246 - INFO - Database connection established successfully.



Query: 'who wrote hamlet'
  [1] 19.9746 — William Shakespeare, the English poet, playwright and dramatist, is the center of adulation and inspiration to artists f...
  [2] 19.8127 — William Shakespeare, also known as the Bard, is responsible for some of the best plays and poetry ever written in the En...
  [3] 19.6336 — If the play dates to 1600, it is most likely to have been first performed by the Lord Chamberlain’s Men at the Globe. Th...


2026-03-07 13:40:29,557 - INFO - SPLADE search: 211576 candidates, top-3 returned (encode=53ms, retrieval=11471ms, scoring=787ms, total=12310ms)



Query: 'symptoms of diabetes'
  [1] 22.9630 — Diabetes symptoms occur when blood sugar (glucose) levels in the body become abnormally elevated. The most common sympto...
  [2] 22.9136 — Individuals can experience different signs and symptoms of diabetes, and sometimes there may be no signs. Some of the si...
  [3] 22.9136 — Individuals can experience different signs and symptoms of diabetes, and sometimes there may be no signs. Some of the si...


## 4) Evaluation — MRR@10

Compute **Mean Reciprocal Rank @ 10** using the ground-truth `qrels` table.

In [8]:
import random
from src.database.connection import get_connection
from src.splade.search import search_gin
from src.splade.encoder import SpladeEncoder

conn = get_connection()
cursor = conn.cursor()

# Get queries that have at least one relevant passage (relevance = 1)
cursor.execute("""
    SELECT DISTINCT q.id, q.text
    FROM queries q
    JOIN qrels qr ON qr.query_id = q.id
    WHERE qr.relevance = 1
""")
eval_queries = cursor.fetchall()

# Sample a subset for faster evaluation
SAMPLE_SIZE = 20
if len(eval_queries) > SAMPLE_SIZE:
    random.seed(42)
    eval_queries = random.sample(eval_queries, SAMPLE_SIZE)

print(f"Evaluating on {len(eval_queries)} queries...\n")

encoder = SpladeEncoder()  # load once
reciprocal_ranks = []

for idx, (query_id, query_text) in enumerate(eval_queries):
    # Relevant passage IDs for this query
    cursor.execute(
        "SELECT passage_id FROM qrels WHERE query_id = %s AND relevance = 1",
        (query_id,)
    )
    relevant_ids = {row[0] for row in cursor.fetchall()}

    # Run SPLADE search
    results = search_gin(
        query_text, top_k=10, conn=conn, encoder=encoder, log_search=False
    )

    # Compute reciprocal rank
    rr = 0.0
    for rank, r in enumerate(results, 1):
        if r["passage_id"] in relevant_ids:
            rr = 1.0 / rank
            break
    reciprocal_ranks.append(rr)

    if (idx + 1) % 50 == 0:
        current_mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
        print(f"  [{idx+1}/{len(eval_queries)}] Running MRR@10 = {current_mrr:.4f}")

mrr_at_10 = sum(reciprocal_ranks) / len(reciprocal_ranks)
print(f"\n{'='*50}")
print(f"Final MRR@10 = {mrr_at_10:.4f}  ({len(eval_queries)} queries)")
print(f"{'='*50}")

cursor.close()
conn.close()

2026-03-07 13:54:20,030 - INFO - Database connection established successfully.
2026-03-07 13:54:20,180 - INFO - Loading SPLADE model 'naver/splade-cocondenser-ensembledistil' on cpu ...
2026-03-07 13:54:20,351 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-07 13:54:20,358 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/config.json "HTTP/1.1 200 OK"


Evaluating on 20 queries...



2026-03-07 13:54:20,461 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-07 13:54:20,466 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/naver/splade-cocondenser-ensembledistil/49cf4c7b0db5b870a401ddf5e2669993ef3699c7/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-07 13:54:20,575 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocondenser-ensembledistil/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-07 13:54:20,677 - INFO - HTTP Request: GET https://huggingface.co/api/models/naver/splade-cocondenser-ensembledistil/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-07 13:54:20,848 - INFO - HTTP Request: HEAD https://huggingface.co/naver/splade-cocondenser-ensembledistil/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-07 13:54:20,856 -


Final MRR@10 = 0.3386  (20 queries)


## Notes

- The first run downloads the model from Hugging Face (~400 MB). Set `HF_TOKEN` in `.env` if access is gated.
- For large-scale indexing, a GPU significantly speeds up encoding.
- The GIN search is fast for small-to-medium collections. For millions of passages, consider an inverted index in Elasticsearch or a dedicated sparse vector store.
- You can re-run indexing at any time; already-indexed passages are skipped automatically.